# QAOA backend comparison

A self-contained study of one read matrix across ideal simulation, calibrated Berlin/Aachen noise and optionally the real devices. Run the settings and sections in order. Each new run stores its circuits, configuration, raw counts, plots and progress locally; no hardware job is submitted unless `RUN_REAL` is true.

Local ideal training is frozen before screening. The strongest method/depth in noisy screening is used for the separate, equal-budget technique study. This is an exploratory result for one problem and calibration, not a universal ranking.

## 1. Imports and user controls
Choose the project kernel and edit the labelled settings before running. `MODE="load"` reads saved results offline; `MODE="resume"` continues a version 2 run using its original settings.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import itertools, json, math, os, sys, time, zlib
from collections import deque
import numpy as np
import pandas as pd
import networkx as nx
from IPython.display import display, Image, Markdown
from scipy.optimize import minimize
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.ticker import PercentFormatter
from matplotlib.colors import PowerNorm
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, qpy
from qiskit.circuit import Delay, Parameter, ParameterVector, pauli_twirl_2q_gates
from qiskit.circuit.library import XGate
from qiskit.quantum_info import Statevector
from qiskit.transpiler import PassManager, generate_preset_pass_manager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDelay, PadDynamicalDecoupling
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.noise.passes import RelaxationNoisePass
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
print({name: metadata.version(name) for name in ('qiskit','qiskit-aer','qiskit-ibm-runtime','numpy','scipy','pandas')})

In [ ]:
# Run the local study, load a saved folder, or resume a version 2 run.
MODE = 'run'              # 'run', 'load', 'resume'
RUN_FOLDER = None         # required for load/resume
RUN_REAL = False          # True explicitly submits real hardware jobs
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'bench'/'all_together.ipynb').exists())
OUTPUT_DIR = ROOT/'bench'/'results'/'backend_comparison'
BACKENDS = ['ibm_berlin', 'ibm_aachen']
READS = [[0,1,1,-1,-1],[1,0,0,-1,-1],[-1,1,1,0,-1],[-1,0,0,1,-1],
         [-1,-1,1,0,1],[-1,-1,0,1,0],[0,-1,-1,0,1],[1,-1,-1,1,0],[0,0,-1,-1,-1]]
METHODS = ['standard', 'warm-start', 'multi-angle']
DEPTHS = [1,2,3]
OBJECTIVE = 'A baseline'
PRUNE_RULE = 'mstforest'  # or 'none'
SHOTS, SCREEN_SHOTS, REPEATS, SCREEN_REPEATS = 2048, 1024, 3, 3
TECHNIQUE_SHOTS = 4096    # total per technique arm, including calibration/folding
TECHNIQUES = ['readout','dd','twirling','zne']  # raw reference always included
STANDARD_ITERATIONS, MULTI_ITERATIONS, STARTS = 100, 500, 5
SEED, TRAINING_SEED, TRANSPILE_SEED, LOCAL_SEED = 71, 71, 71, 17
OPTIMIZATION_LEVEL, LOCAL_BUDGET, LOCAL_RESTARTS, EPSILON = 2, 128, 16, 0.25
SHOW_PLOTS = True
ALL_DISTRIBUTION_DEPTHS = False
BACKEND_OVERRIDES = None  # optional injected fake backends for local tests
CONFIG = {'mode':MODE,'run_folder':str(RUN_FOLDER) if RUN_FOLDER is not None else None,
          'run_real':RUN_REAL,'output_dir':str(OUTPUT_DIR),'backends':BACKENDS,'reads':READS,
          'methods':METHODS,'depths':DEPTHS,'objective':OBJECTIVE,'prune_rule':PRUNE_RULE,'shots':SHOTS,
          'screen_shots':SCREEN_SHOTS,'repeats':REPEATS,'screen_repeats':SCREEN_REPEATS,
          'technique_shots':TECHNIQUE_SHOTS,'techniques':TECHNIQUES,'standard_iterations':STANDARD_ITERATIONS,
          'multi_iterations':MULTI_ITERATIONS,'starts':STARTS,'seed':SEED,'training_seed':TRAINING_SEED,
          'transpile_seed':TRANSPILE_SEED,'local_seed':LOCAL_SEED,'optimization_level':OPTIMIZATION_LEVEL,
          'local_budget':LOCAL_BUDGET,'local_restarts':LOCAL_RESTARTS,'epsilon':EPSILON,
          'all_distribution_depths':ALL_DISTRIBUTION_DEPTHS}
print('Run mode:', MODE, '| hardware submission:', RUN_REAL)
display(pd.DataFrame([CONFIG]).T.rename(columns={0:'setting'}))

## 2. Evidence and exact references
Same/opposite pair votes define the full graph. Full pairwise and MEC minima judge every sample. State labels list SNP 1 first; Qiskit count strings are reversed when decoded.

In [ ]:
def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)


def ibm_service(root):
    values = {}
    env_file = Path(root) / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8').splitlines():
            if '=' in line and not line.lstrip().startswith('#'):
                key, val = line.split('=', 1)
                values[key.strip()] = val.strip().strip("\"'")
    token = os.getenv('IBM_API_KEY') or values.get('IBM_API_KEY')
    instance = (os.getenv('IBM_INSTANCE_CRN') or os.getenv('IBM_INSTANCE')
                or values.get('IBM_INSTANCE_CRN') or values.get('IBM_INSTANCE'))
    if not token:
        raise ValueError('Set IBM_API_KEY in the environment or project .env file.')
    options = {'channel': 'ibm_quantum_platform', 'token': token}
    if instance:
        options['instance'] = instance
    return QiskitRuntimeService(**options)


def problem(reads):
    reads = np.asarray(reads)
    if reads.ndim != 2 or not 1 <= reads.shape[1] <= 12 or not len(reads) or not np.isin(reads, [-1, 0, 1]).all():
        raise ValueError('Reads must be a nonempty matrix of -1/0/1 with 1–12 SNPs.')
    n = reads.shape[1]
    a, d = np.zeros((n,n)), np.zeros((n,n))
    for row in reads:
        for i,j in itertools.combinations(np.flatnonzero(row != -1), 2):
            (a if row[i] == row[j] else d)[i,j] += 1
    states = np.array(list(itertools.product([0,1], repeat=n)), dtype=int)
    full = sum((np.where(states[:,i] == states[:,j], d[i,j], a[i,j])
                for i,j in itertools.combinations(range(n),2)), np.zeros(len(states)))
    mec = np.zeros(len(states))
    for row in reads:
        seen = row != -1
        dist = (states[:,seen] != row[seen]).sum(axis=1)
        mec += np.minimum(dist, seen.sum()-dist)
    return reads, a, d, states, full, mec


def forest_j(a, d):
    # Same descending |A-D| Kruskal forest as the combined notebook.
    n = len(a); parent = list(range(n)); j = np.zeros_like(a)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    edges = [(i,k) for i,k in itertools.combinations(range(n),2) if a[i,k] != d[i,k]]
    edges.sort(key=lambda e: abs(a[e]-d[e]), reverse=True)
    for i,k in edges:
        x,y = find(i), find(k)
        if x != y:
            parent[x] = y; j[i,k] = (d[i,k]-a[i,k])/2
    return j


def measurements(circuit):
    return {circuit.find_bit(item.clbits[0]).index: circuit.find_bit(item.qubits[0]).index
            for item in circuit.data if item.operation.name == 'measure'}


def calibrations(compiled):
    mapping = measurements(compiled)
    n = compiled.num_clbits
    if set(mapping) != set(range(n)):
        raise ValueError('Expected exactly one measured physical qubit per SNP.')
    circuits = []
    for prepared in (0,1):
        qc = QuantumCircuit(QuantumRegister(compiled.num_qubits, 'q'), ClassicalRegister(n, 'meas'), name=f'calibration_{prepared}')
        for bit, qubit in mapping.items():
            if prepared: qc.x(qubit)
            qc.measure(qubit, bit)
        circuits.append(qc)
    return circuits


def circuit_stats(qc):
    two = lambda inst: inst.operation.num_qubits == 2 and inst.operation.name != 'barrier'
    return {'depth': qc.depth(), 'two_qubit_depth': qc.depth(filter_function=two),
            'two_qubit_gates': sum(two(i) for i in qc.data),
            'physical_qubits_used': len({qc.find_bit(q).index for inst in qc.data
                                         if inst.operation.name != 'barrier' for q in inst.qubits}),
            'operations': dict(qc.count_ops()), 'measurement_map': measurements(qc)}


def probability_metrics(counts, states, full, mec, ideal_probs):
    n = states.shape[1]; total = sum(counts.values())
    if not total: raise ValueError('No counts returned')
    index = {tuple(s): i for i,s in enumerate(states)}
    probs = np.zeros(len(states))
    for key,count in counts.items():
        bits = tuple(int(b) for b in key.replace(' ','')[::-1])
        if len(bits) != n: raise ValueError(f'Unexpected measurement width: {key}')
        probs[index[bits]] += count/total
    success = float(probs[np.isclose(full, full.min(), rtol=0, atol=1e-9)].sum())
    return {'shots': total, 'full_optimum_probability': success,
            'full_optimum_standard_error': float(np.sqrt(success*(1-success)/total)),
            'mean_full_cost': float(probs @ full),
            'mec_optimum_probability': float(probs[np.isclose(mec,mec.min(),rtol=0,atol=1e-9)].sum()),
            'mean_mec': float(probs @ mec),
            'total_variation_from_ideal': float(np.abs(probs-ideal_probs).sum()/2)}


def correction(counts, cal_counts, a, d):
    n = len(a); response = np.zeros((n,2,2))
    for prepared, data in enumerate(cal_counts):
        total = sum(data.values())
        for key,count in data.items():
            for i,bit in enumerate(key.replace(' ','')[::-1]):
                response[i,int(bit),prepared] += count/total
    conditions = [float(np.linalg.cond(m)) for m in response]
    if not np.isfinite(conditions).all() or max(conditions) > 1e6:
        raise ValueError('Readout calibration is singular or unstable.')
    z = np.array([1.,-1.]) @ np.linalg.inv(response)
    total = sum(counts.values()); estimate = 0.
    for key,count in counts.items():
        bits = [int(b) for b in key.replace(' ','')[::-1]]
        for i,k in itertools.combinations(range(n),2):
            estimate += count*((a[i,k]+d[i,k])/2 + (d[i,k]-a[i,k])/2*z[i,bits[i]]*z[k,bits[k]])
    return float(estimate/total), response.tolist(), conditions


## 3. Pruning, local search and circuit families
The forest keeps the strongest connectivity-preserving edges. Warm-start uses the local-search candidate, never hidden truth or an enumerated solution. Circuit implementations below are from the combined notebook.

In [ ]:
def _sym(J):
    return J + J.T


def _greedy(W, k, rng):
    """Randomised greedy: visit SNPs in random order, give each the letter that best fits so far."""
    n = len(W)
    order = np.argsort(rng.random((k, n)), axis=1)
    Z, ar = np.zeros((k, n)), np.arange(k)
    for t in range(n):
        i = order[:, t]
        fld = (W[i] * Z).sum(axis=1)
        tie = rng.choice([-1.0, 1.0], size=k)
        Z[ar, i] = np.where(fld > 0, -1.0, np.where(fld < 0, 1.0, tie))
    return Z


def greedy_local_search(J, offset, budget, runs, rng, target=-math.inf):
    """Greedy start + steepest 1-flip descent, restarting when stuck, until the budget is spent."""
    n, W = len(J), _sym(J)
    ar = np.arange(runs)
    Z = _greedy(W, runs, rng)
    used = np.full(runs, 2 * n)
    h = Z @ W
    E = offset + 0.5 * (Z * h).sum(axis=1)
    best, bestZ = E.copy(), Z.copy()
    hit = np.where(E <= target + 1e-9, used.astype(float), np.inf)
    while True:
        active = used + n <= budget
        if not active.any():
            break
        dE = -2.0 * Z * h                                  # cost change of every single flip
        j = dE.argmin(axis=1)
        m = dE[ar, j]
        used[active] += n
        imp = active & (m < -1e-12)
        if imp.any():
            idx, jj = ar[imp], j[imp]
            z_old = Z[idx, jj].copy()
            Z[idx, jj] = -z_old
            h[idx] -= 2.0 * z_old[:, None] * W[jj]
            E[imp] += m[imp]
        stuck = active & ~imp
        restart = stuck & (used + 2 * n <= budget)
        if restart.any():
            Z[restart] = _greedy(W, int(restart.sum()), rng)
            used[restart] += 2 * n
            h[restart] = Z[restart] @ W
            E[restart] = offset + 0.5 * (Z[restart] * h[restart]).sum(axis=1)
        newly = (E <= target + 1e-9) & np.isinf(hit)
        hit[newly] = used[newly]
        used[stuck & ~restart] = budget
        better = E < best - 1e-12
        best[better], bestZ[better] = E[better], Z[better]
    return best, ((1 - bestZ) / 2).astype(np.int8), np.minimum(used, budget), hit


def qaoa_scale(J):
    return float(max(1, np.max(np.abs(J))))


def make_qaoa(J, p=1):
    """Same circuit as the notebook's make_qaoa: H, then per layer RZZ on active pairs and RX."""
    from qiskit import QuantumCircuit
    from qiskit.circuit import ParameterVector

    J = np.asarray(J, float)
    n = J.shape[0]
    scale = qaoa_scale(J)
    theta = ParameterVector("theta", 2 * p)
    gammas, betas = theta[:p], theta[p:]
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p):
        for a in range(n):
            for b in range(a + 1, n):
                if J[a, b] != 0:
                    qc.rzz(2 * gammas[layer] * J[a, b] / scale, a, b)
        for q in range(n):
            qc.rx(2 * betas[layer], q)
    return qc, theta, scale


def warm_start_qaoa(n, J, c, p, eps):
    J = np.asarray(J, dtype=float)
    c = np.clip(c, eps, 1 - eps)
    theta = 2 * np.arcsin(np.sqrt(c))
    scale = float(max(1, np.max(np.abs(J))))
    gam = [Parameter(f'g{k}') for k in range(p)]
    bet = [Parameter(f'b{k}') for k in range(p)]
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.ry(theta[i], i)                       # warm-start state
    for k in range(p):
        for i in range(n):
            for j in range(i + 1, n):
                if J[i, j] != 0:
                    qc.rzz(2 * gam[k] * J[i, j] / scale, i, j)
        for i in range(n):                       # warm-start mixer
            qc.ry(-theta[i], i)
            qc.rz(-2 * bet[k], i)
            qc.ry(theta[i], i)
    return qc, list(gam) + list(bet)


def make_ma_qaoa(J, p):
    J = np.asarray(J, dtype=float); n = J.shape[0]
    scale = float(max(1, np.max(np.abs(J))))
    active = [(a, b) for a in range(n) for b in range(a + 1, n) if J[a, b] != 0]
    per_layer = len(active) + n
    theta = ParameterVector('ma', p * per_layer)
    qc = QuantumCircuit(n); qc.h(range(n))
    for layer in range(p):
        base = layer * per_layer
        for idx, (a, b) in enumerate(active):
            qc.rzz(2 * theta[base + idx] * J[a, b] / scale, a, b)
        for q in range(n):
            qc.rx(2 * theta[base + len(active) + q], q)
    return qc, list(theta)


In [ ]:
def native_scores(states, j, offset):
    z = 1 - 2 * np.asarray(states)
    return offset + sum((j[i, k] * z[:, i] * z[:, k]
                         for i, k in itertools.combinations(range(len(j)), 2)),
                        np.zeros(len(states)))


def make_template(method, j, depth, seed_bits, epsilon):
    if method == 'standard':
        circuit, params, _ = make_qaoa(j, depth)
    elif method == 'warm-start':
        circuit, params = warm_start_qaoa(len(j), j, seed_bits, depth, epsilon)
    elif method == 'multi-angle':
        circuit, params = make_ma_qaoa(j, depth)
    else:
        raise ValueError(f'Unknown QAOA method: {method}')
    return circuit, list(params)


def fit_template(circuit, params, states, native, full, maxiter, starts, seed):
    if len(set(params)) != len(params) or not set(circuit.parameters).issubset(params):
        raise ValueError('Circuit parameters disagree with their ordered list.')
    active = [p for p in params if p in circuit.parameters]
    indices = [sum(int(bit) << i for i, bit in enumerate(state)) for state in states]
    native_basis = np.zeros(len(states)); full_basis = np.zeros(len(states))
    native_basis[indices] = native; full_basis[indices] = full
    rng = np.random.default_rng(seed); trials = []; began = time.perf_counter()
    if not active:
        probabilities = Statevector.from_instruction(circuit).probabilities()
        return circuit.copy(), {'parameters': [], 'angles': [], 'starts': [], 'evaluations': 1,
             'best_native_cost': float(probabilities @ native_basis), 'wall_seconds': time.perf_counter()-began}
    for start in range(starts):
        trace = []
        def objective(values):
            probabilities = Statevector.from_instruction(circuit.assign_parameters(dict(zip(active, values)))).probabilities()
            native_value = float(probabilities @ native_basis)
            trace.append({'evaluation': len(trace)+1, 'native_cost': native_value,
                          'full_cost': float(probabilities @ full_basis)})
            return native_value
        result = minimize(objective, rng.uniform(0, np.pi, len(active)), method='COBYLA',
                          options={'maxiter': maxiter})
        trials.append({'start': start, 'angles': np.asarray(result.x, float).tolist(),
                       'native_cost': float(result.fun), 'evaluations': int(result.nfev),
                       'success': bool(result.success), 'message': str(result.message), 'trace': trace})
    best = min(trials, key=lambda trial: trial['native_cost'])
    frozen = circuit.assign_parameters(dict(zip(active, best['angles'])))
    return frozen, {'parameters': [str(p) for p in active], 'angles': best['angles'],
                    'starts': trials, 'best_start': best['start'],
                    'best_native_cost': best['native_cost'],
                    'evaluations': sum(t['evaluations'] for t in trials),
                    'wall_seconds': time.perf_counter()-began}


def ideal_probabilities(circuit, states):
    vector = Statevector.from_instruction(circuit.remove_final_measurements(inplace=False)).probabilities()
    return np.array([vector[sum(int(bit) << i for i, bit in enumerate(state))] for state in states])


def save_circuit(path, circuit):
    with Path(path).open('wb') as f:
        qpy.dump(circuit, f)


def load_circuit(path):
    with Path(path).open('rb') as f:
        return qpy.load(f)[0]


def seeded(config, *parts):
    return int((config['seed'] + zlib.crc32('|'.join(map(str, parts)).encode())) % (2**32-1))


def trial_key(stage, backend, method, depth, repeat, technique, variant):
    return '|'.join(map(str, (stage, backend, method, depth, repeat, technique, variant)))


def workload(config):
    n = len(config['methods']) * len(config['depths'])
    shots = config['shots']; tech = config['technique_shots']; repeats = config['repeats']
    counts = {'raw': n*shots, 'raw_reference': tech}
    for name in ('readout', 'dd', 'twirling', 'zne'):
        if name in config['techniques']:
            counts[name] = tech
    per_backend = repeats * sum(counts.values())
    return {'screening_simulator_shots': len(config['backends'])*n*config['screen_shots']*config['screen_repeats'],
            'ideal_shots': n*shots*repeats, 'noise_shots': per_backend*len(config['backends']),
            'hardware_shots_if_enabled': per_backend*len(config['backends']) if config['run_real'] else 0,
            'hardware_shots_full_plan': per_backend*len(config['backends']),
            'hardware_jobs_before_skips': len(config['backends'])*repeats if config['run_real'] else 0,
            'per_backend_per_repeat': counts}


def validate_config(config):
    if config['objective'] != 'A baseline':
        raise ValueError('Only the A baseline objective is supported.')
    if config['mode'] not in ('run', 'load', 'resume'):
        raise ValueError('MODE must be run, load or resume.')
    if type(config['run_real']) is not bool:
        raise ValueError('RUN_REAL must be a boolean.')
    if not config['methods'] or not set(config['methods']) <= {'standard','warm-start','multi-angle'}:
        raise ValueError('Select at least one supported method.')
    if not config['backends'] or len(set(config['backends'])) != len(config['backends']):
        raise ValueError('Select distinct backend names.')
    if not config['depths'] or any(type(p) is not int or p < 1 for p in config['depths']):
        raise ValueError('Depths must be positive integers.')
    for key in ('shots','screen_shots','repeats','screen_repeats','starts','standard_iterations','multi_iterations','technique_shots','local_budget','local_restarts'):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError(f'{key} must be a positive integer.')
    if config['technique_shots'] % 16:
        raise ValueError('TECHNIQUE_SHOTS must be divisible by 16.')
    if not set(config['techniques']) <= {'readout','dd','twirling','zne'}:
        raise ValueError('Unknown noise technique.')
    if config['mode'] != 'run' and not config['run_folder']:
        raise ValueError('Set RUN_FOLDER for load or resume.')
    problem(config['reads'])


## 4. Compilation and noise techniques
Each arm has the same total shot budget. Readout correction and ZNE estimate cost only. DD requires verified delay relaxation; unsupported or unverified transformations are recorded as skips.

In [ ]:
def compact_probabilities(circuit, n):
    """Logical classical-bit probabilities using only physically active qubits."""
    mapping = measurements(circuit)
    active = sorted({circuit.find_bit(q).index for item in circuit.data
                     if item.operation.name not in ('barrier', 'delay') for q in item.qubits})
    if len(active) > 12:
        raise ValueError(f'Cannot verify ideal equivalence: {len(active)} active qubits exceed 12.')
    remap = {physical: i for i, physical in enumerate(active)}
    small = QuantumCircuit(len(active))
    for item in circuit.data:
        if item.operation.name in ('barrier', 'delay', 'measure'):
            continue
        if item.clbits:
            raise ValueError('Classical control is unsupported in equivalence verification.')
        small.append(item.operation, [remap[circuit.find_bit(q).index] for q in item.qubits])
    vector = Statevector.from_instruction(small).probabilities(); output = np.zeros(2**n)
    for basis, probability in enumerate(vector):
        logical = sum(((basis >> remap[physical]) & 1) << bit for bit, physical in mapping.items())
        output[logical] += probability
    return output


def fold_two_qubit(circuit, factor, backend):
    if factor not in (1, 3):
        raise ValueError('Only factors 1 and 3 are supported.')
    folded = circuit.copy_empty_like()
    for item in circuit.data:
        gate = item.operation
        if gate.num_qubits == 2 and gate.name != 'barrier' and factor == 3:
            inverse = gate.inverse()
            if inverse.name not in backend.target.operation_names:
                raise ValueError(f'Inverse gate {inverse.name} is not in the backend target.')
            folded.append(gate, item.qubits, item.clbits)
            folded.append(inverse, item.qubits, item.clbits)
        folded.append(gate, item.qubits, item.clbits)
    return folded


def delay_relaxation_available(simulator):
    model = simulator.options.noise_model
    for noise_pass in getattr(model, '_custom_noise_passes', []):
        ops = getattr(noise_pass, '_ops', ()) or ()
        if isinstance(noise_pass, RelaxationNoisePass) and Delay in ops:
            return True
    return False


def prepare_techniques(compiled, backend, simulator, config, repeat):
    """Return circuits and shot allocations. Every non-skipped arm costs TECHNIQUE_SHOTS."""
    budget = config['technique_shots']; n = compiled.num_clbits
    try:
        reference = compact_probabilities(compiled, n)
    except ValueError as exc:
        return {'raw_reference':[('target',compiled,budget)]}, {name:f'Equivalence verification unavailable: {exc}' for name in config['techniques']}
    base = compiled; dd = None; skips = {}
    if 'dd' in config['techniques']:
        try:
            if not delay_relaxation_available(simulator):
                raise ValueError('No verified delay-relaxation pass in calibrated noise model.')
            base = PassManager([ALAPScheduleAnalysis(target=backend.target),
                                PadDelay(target=backend.target)]).run(compiled)
            dd = PassManager([ALAPScheduleAnalysis(target=backend.target),
                              PadDynamicalDecoupling(dd_sequence=[XGate(),XGate()], qubits=sorted({compiled.find_bit(q).index for item in compiled.data if item.operation.name not in ('barrier','delay') for q in item.qubits}), target=backend.target)]).run(compiled)
            if not np.allclose(compact_probabilities(base,n), reference, atol=1e-8):
                raise ValueError('Scheduled baseline changed ideal probabilities.')
            if not np.allclose(compact_probabilities(dd,n), reference, atol=1e-8):
                raise ValueError('DD changed ideal probabilities.')
        except Exception as exc:
            base = compiled; dd = None; skips['dd'] = f'{type(exc).__name__}: {exc}'
    arms = {'raw_reference': [('target', base, budget)]}
    if 'readout' in config['techniques']:
        try:
            cal = calibrations(compiled)
            pm = generate_preset_pass_manager(backend=backend, optimization_level=0,
                  initial_layout=list(range(backend.num_qubits)), seed_transpiler=config['transpile_seed'])
            cal = pm.run(cal)
            if any(measurements(c) != measurements(compiled) for c in cal):
                raise ValueError('Calibration measurement mapping changed.')
            arms['readout'] = [('target',base,budget//2),('cal0',cal[0],budget//4),('cal1',cal[1],budget//4)]
        except Exception as exc:
            skips['readout'] = f'{type(exc).__name__}: {exc}'
    if dd is not None:
        arms['dd'] = [('target',dd,budget)]
    if 'twirling' in config['techniques']:
        try:
            variants = pauli_twirl_2q_gates(base, seed=seeded(config, backend.name, repeat, 'twirl'),
                                           num_twirls=16, target=backend.target)
            for circuit in variants:
                if measurements(circuit) != measurements(compiled) or not np.allclose(compact_probabilities(circuit,n),reference,atol=1e-8):
                    raise ValueError('Twirled circuit changed logical measurement probabilities.')
            arms['twirling'] = [(f'twirl_{i}',circuit,budget//16) for i,circuit in enumerate(variants)]
        except Exception as exc:
            skips['twirling'] = f'{type(exc).__name__}: {exc}'
    if 'zne' in config['techniques']:
        try:
            folded = fold_two_qubit(base, 3, backend)
            if not np.allclose(compact_probabilities(folded,n),reference,atol=1e-8):
                raise ValueError('Folding changed ideal probabilities.')
            arms['zne'] = [('factor_1',base,budget//2),('factor_3',folded,budget//2)]
        except Exception as exc:
            skips['zne'] = f'{type(exc).__name__}: {exc}'
    for arm in list(arms):
        try:
            for _,circuit,_ in arms[arm]:
                for item in circuit.data:
                    if item.operation.name == 'barrier':
                        continue
                    qargs = tuple(circuit.find_bit(q).index for q in item.qubits)
                    if not backend.target.instruction_supported(operation_name=item.operation.name, qargs=qargs):
                        raise ValueError(f'Unsupported target instruction {item.operation.name} on {qargs}')
        except Exception as exc:
            if arm == 'raw_reference':
                raise
            skips[arm] = f'{type(exc).__name__}: {exc}'
            del arms[arm]
    for arm, variants in arms.items():
        if sum(v[2] for v in variants) != budget:
            raise AssertionError(f'{arm} has the wrong shot budget')
    return arms, skips


def normalize_counts(counts, states):
    n = states.shape[1]; indices = {tuple(s): i for i,s in enumerate(states)}
    probs = np.zeros(len(states)); total = sum(counts.values())
    if total <= 0:
        raise ValueError('Empty counts.')
    for key, count in counts.items():
        bits = tuple(map(int,key.replace(' ','')[::-1]))
        if len(bits) != n:
            raise ValueError(f'Unexpected measurement width: {key}')
        probs[indices[bits]] += count/total
    return probs


def merge_counts(counts_list):
    merged = {}
    for counts in counts_list:
        for key, count in counts.items():
            merged[key] = merged.get(key, 0) + int(count)
    return merged


def result_row(task, counts, states, full, mec, ideal):
    metrics = probability_metrics(counts,states,full,mec,ideal)
    return {**{k: task[k] for k in ('stage','backend','method','depth','repeat','technique')}, **metrics}


## 5. Persistence and recovery
Version 2 stores individual raw records, circuits, settings and submitted job/PUB associations before result processing. Resume retrieves submitted jobs rather than resubmitting them.

In [ ]:
import hashlib
import shutil


def write_manifest(path, manifest):
    save_json(path/'manifest.json', manifest)


def initialize_run(root, config):
    if config['mode'] == 'run':
        path = Path(config['output_dir'])/datetime.now().astimezone().strftime('%Y%m%d_%H%M%S_%f')
        path.mkdir(parents=True, exist_ok=False)
        for folder in ('circuits','records','plots'):
            (path/folder).mkdir()
        snapshot = Path(root)/'bench'/'backend_comparison.ipynb'
        if snapshot.exists():
            shutil.copyfile(snapshot,path/'notebook_snapshot.ipynb')
        manifest = {'schema_version':2, 'status':'started', 'created_utc':datetime.now(timezone.utc).isoformat(),
                    'config':config, 'jobs':[], 'skips':[], 'champion':None,
                    'versions':{name:metadata.version(name) for name in ('qiskit','qiskit-aer','qiskit-ibm-runtime','numpy','scipy','pandas')}}
        write_manifest(path,manifest)
        return path, manifest
    path = Path(config['run_folder'])
    manifest = json.loads((path/'manifest.json').read_text(encoding='utf8'))
    if config['mode'] == 'resume' and manifest.get('schema_version') != 2:
        raise ValueError('Resume requires a version 2 run; older runs can be loaded.')
    return path, manifest


def record_path(path, key):
    return path/'records'/f'{hashlib.sha256(key.encode()).hexdigest()}.json'


def circuit_path(path, key):
    return path/'circuits'/f'{hashlib.sha256(key.encode()).hexdigest()}.qpy'


def existing_record(path, key):
    file = record_path(path,key)
    return json.loads(file.read_text(encoding='utf8')) if file.exists() else None


def store_result(path, task, counts, extra):
    payload = {'key':task['key'], 'stage':task['stage'], 'backend':task['backend'],
               'method':task['method'], 'depth':task['depth'], 'repeat':task['repeat'],
               'technique':task['technique'], 'variant':task['variant'], 'shots':task['shots'],
               'circuit_file':str(circuit_path(path,task['key']).relative_to(path)),
               'circuit_resources':task['resources'], 'counts':{str(k):int(v) for k,v in counts.items()}, **extra}
    if sum(payload['counts'].values()) != task['shots']:
        raise ValueError(f"Unexpected shot count for {task['key']}")
    save_json(record_path(path,task['key']),payload)


def task_for(path, circuit, stage, backend, method, depth, repeat, technique, variant, shots):
    key = trial_key(stage, backend, method, depth, repeat, technique, variant)
    filename = circuit_path(path,key)
    if not filename.exists():
        save_circuit(filename,circuit)
    return {'key':key,'circuit':circuit,'stage':stage,'backend':backend,'method':method,
            'depth':depth,'repeat':repeat,'technique':technique,'variant':variant,'shots':shots,
            'resources':circuit_stats(circuit)}


def run_local_tasks(path, simulator, tasks, config):
    pending = [task for task in tasks if existing_record(path,task['key']) is None]
    for shots in sorted({task['shots'] for task in pending}):
        group = [task for task in pending if task['shots'] == shots]
        if not group: continue
        seed = seeded(config, group[0]['stage'], group[0]['backend'], group[0]['repeat'], shots)
        began = time.perf_counter()
        result = simulator.run([t['circuit'] for t in group], shots=shots, seed_simulator=seed).result()
        elapsed = time.perf_counter()-began
        for i, task in enumerate(group):
            store_result(path,task,result.get_counts(i),{'seed_simulator':seed,'batch_wall_seconds':elapsed,
                                                       'batch_circuits':len(group),'batch_task_keys':[t['key'] for t in group], 'job_id':None,'pub_index':i})


def run_remote_tasks(path, manifest, service, backend, tasks):
    """Reuse recorded submissions, including when interrupted while collecting results."""
    expected = [task['key'] for task in tasks]
    if not expected or all(existing_record(path,k) for k in expected): return
    previous = next((j for j in manifest['jobs'] if j['backend']==backend.name and j['task_keys']==expected),None)
    if previous is None:
        if any(existing_record(path,k) for k in expected):
            raise RuntimeError('Partial records have no matching job; refusing to resubmit.')
        sampler = SamplerV2(mode=backend)
        sampler.options.dynamical_decoupling.enable = False
        sampler.options.twirling.enable_gates = False
        sampler.options.twirling.enable_measure = False
        pubs = [(t['circuit'],None,t['shots']) for t in tasks]
        began = time.perf_counter(); job = sampler.run(pubs)
        previous = {'backend':backend.name,'job_id':job.job_id(),'task_keys':expected,
                    'status':'submitted','submitted_utc':datetime.now(timezone.utc).isoformat(),
                    'tasks':[{k:v for k,v in t.items() if k != 'circuit'} for t in tasks]}
        manifest['jobs'].append(previous); write_manifest(path,manifest)
        print('Submitted:',backend.name,previous['job_id'],flush=True)
    else:
        began = time.perf_counter(); job = service.job(previous['job_id'])
    result = job.result(); elapsed = time.perf_counter()-began
    for i, task in enumerate(tasks):
        if existing_record(path,task['key']) is None:
            store_result(path,task,result[i].data.meas.get_counts(),
                         {'job_id':previous['job_id'],'pub_index':i,'batch_wall_seconds':elapsed,
                          'batch_circuits':len(tasks)})
    previous['status'] = 'completed'; write_manifest(path,manifest)


def provenance(backend):
    properties = backend.properties() if callable(getattr(backend,'properties',None)) else None
    if properties is None:
        return {'backend':backend.name,'properties':None}
    return {'backend':backend.name,'properties':properties.to_dict()}


def calibration_stamp(data):
    props = data.get('properties') or {}
    return str(props.get('last_update_date'))


def setup_backends(root, path, manifest, config, backend_objects=None):
    service = None if backend_objects is not None else ibm_service(root)
    backends = backend_objects if backend_objects is not None else {name:service.backend(name) for name in config['backends']}
    simulators = {}
    for name in config['backends']:
        backend = backends[name]
        if backend.num_qubits < len(config['reads'][0]):
            raise ValueError(f'{name} has insufficient qubits.')
        current = provenance(backend); snapshot = path/f'{name}_calibration.json'
        if snapshot.exists():
            previous = json.loads(snapshot.read_text(encoding='utf8'))
            if calibration_stamp(previous) != calibration_stamp(current):
                raise RuntimeError(f'{name} calibration changed since this run; start a new run for reproducible simulation.')
        else:
            save_json(snapshot,current)
        simulator = AerSimulator.from_backend(backend,method='matrix_product_state',max_parallel_threads=1)
        simulators[name]=simulator
        info = {'backend':name,'captured_utc':datetime.now(timezone.utc).isoformat(),
                'delay_relaxation_verified':delay_relaxation_available(simulator),
                'noise_model':NoiseModel.from_backend(backend).to_dict(serializable=True)}
        if not (path/f'{name}_noise.json').exists(): save_json(path/f'{name}_noise.json',info)
    return service,backends,simulators


def read_records(path):
    return [json.loads(f.read_text(encoding='utf8')) for f in sorted((path/'records').glob('*.json'))]


def summarize_records(path, states, full, mec, ideal, a, d):
    records = read_records(path); raw = []; grouped = {}
    for r in records:
        if r['stage'] == 'screen' or (r['stage'] in ('ideal','noisy','real') and r['technique']=='raw'):
            raw.append(result_row(r,r['counts'],states,full,mec,ideal[(r['method'],r['depth'])]))
        elif r['stage'] in ('noisy','real'):
            key=(r['stage'],r['backend'],r['method'],r['depth'],r['repeat'],r['technique'])
            grouped.setdefault(key,{})[r['variant']]=r
    estimates=[]
    for key, items in grouped.items():
        stage,backend,method,depth,repeat,technique=key
        row={'stage':stage,'backend':backend,'method':method,'depth':depth,'repeat':repeat,'technique':technique,
             'total_shots':sum(r['shots'] for r in items.values()),'mean_full_cost':None,'optimum_probability':None,
             'diagnostic':None}
        try:
            expected = {'raw_reference':1,'readout':3,'dd':1,'twirling':16,'zne':2}[technique]
            if len(items) != expected: continue
            if technique=='readout':
                row['mean_full_cost'],_,_=correction(items['target']['counts'],
                    [items['cal0']['counts'],items['cal1']['counts']],a,d)
            elif technique=='zne':
                one=probability_metrics(items['factor_1']['counts'],states,full,mec,ideal[(method,depth)])['mean_full_cost']
                three=probability_metrics(items['factor_3']['counts'],states,full,mec,ideal[(method,depth)])['mean_full_cost']
                row.update(factor_1_cost=one,factor_3_cost=three,mean_full_cost=(3*one-three)/2)
            else:
                samples=merge_counts([item['counts'] for item in items.values()])
                metrics=probability_metrics(samples,states,full,mec,ideal[(method,depth)])
                row.update(mean_full_cost=metrics['mean_full_cost'],optimum_probability=metrics['full_optimum_probability'])
            if row['mean_full_cost'] < float(full.min()) or row['mean_full_cost'] > float(full.max()):
                row['diagnostic']='Estimate lies outside the exact cost range.'
        except (ValueError,KeyError) as exc:
            row['diagnostic']=f'{type(exc).__name__}: {exc}'
        estimates.append(row)
    frame=pd.DataFrame(raw); tech=pd.DataFrame(estimates)
    if not frame.empty: frame.to_csv(path/'results.csv',index=False)
    if not tech.empty: tech.to_csv(path/'techniques.csv',index=False)
    return frame,tech,records


def recover_submitted(path, manifest, service):
    for job in manifest['jobs']:
        if job['status'] == 'completed':
            continue
        if 'tasks' not in job:
            raise ValueError('Pending job lacks task metadata for automatic recovery.')
        result = service.job(job['job_id']).result()
        for i, task in enumerate(job['tasks']):
            if existing_record(path,task['key']) is None:
                store_result(path,task,result[i].data.meas.get_counts(),
                             {'job_id':job['job_id'],'pub_index':i,'batch_wall_seconds':None,
                              'batch_circuits':len(job['tasks'])})
        job['status']='completed'; write_manifest(path,manifest)


## 6. Focused plots
Each figure addresses one question. Wilson intervals describe shot uncertainty, while repeated-run standard deviations are displayed separately. The outcome CSV keeps SNP labels as text; use `dtype={"state_snp_first": str}` when reading it with pandas.

In [ ]:
def wilson(k, n):
    z=1.96; p=k/n; den=1+z*z/n
    center=(p+z*z/(2*n))/den
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0.,center-half),min(1.,center+half)


def figure_file(path, filename, x, y, title, xlabel, ylabel, *, legend=True, vline=None):
    fig=Figure(figsize=(8,4.5),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
    for label,values in y.items():
        if isinstance(values,dict):
            ax.errorbar(x,values['mean'],yerr=values.get('error'),marker='o',capsize=3,label=label)
        else:
            ax.plot(x,values,marker='o',label=label)
    if vline is not None: ax.axhline(vline,ls='--',color='gray',label='Uniform baseline')
    ax.set(title=title,xlabel=xlabel,ylabel=ylabel); ax.grid(alpha=.2)
    if legend and y: ax.legend(fontsize=8)
    fig.savefig(path/'plots'/filename,dpi=150,bbox_inches='tight'); fig.clear()


def plot_study(path,manifest,states,full,mec,training,stats,ideal,frame,tech,records):
    out=path/'plots'; out.mkdir(exist_ok=True)
    config=manifest['config']; problem_data=json.loads((path/'problem.json').read_text(encoding='utf8'))
    champ=manifest['champion']; champ_depth=champ['depth']; outputs=[]
    fig=Figure(figsize=(7,3.5),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
    reads=np.asarray(config['reads']); im=ax.imshow(reads,cmap='coolwarm',vmin=-1,vmax=1,aspect='auto')
    ax.set(title='Shared input reads',xlabel='SNP',ylabel='Read',xticks=range(reads.shape[1]),
           xticklabels=range(1,reads.shape[1]+1)); fig.colorbar(im,ax=ax,label='−1 missing · 0 / 1 observed')
    fig.savefig(out/'01_reads.png',dpi=150); fig.clear()
    graph=nx.Graph(); graph.add_nodes_from(range(len(states[0]))); a=np.array(problem_data['same']); d=np.array(problem_data['opposite'])
    j=np.array(problem_data['native_j']); retained=[]; dropped=[]
    for i,k in itertools.combinations(range(len(j)),2):
        if a[i,k]+d[i,k]:
            graph.add_edge(i,k); (retained if j[i,k] else dropped).append((i,k))
    fig=Figure(figsize=(6,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots(); positions=nx.circular_layout(graph)
    nx.draw_networkx_nodes(graph,positions,ax=ax,node_color='#cfe9ef',node_size=600)
    nx.draw_networkx_labels(graph,positions,labels={i:str(i+1) for i in graph},ax=ax)
    nx.draw_networkx_edges(graph,positions,edgelist=retained,ax=ax,edge_color='#247c8c',width=2)
    nx.draw_networkx_edges(graph,positions,edgelist=dropped,ax=ax,edge_color='#aeb9c0',style='dashed')
    ax.set_title('Retained edges (solid) and removed edges (dashed)'); ax.set_axis_off()
    fig.savefig(out/'02_graph.png',dpi=150); fig.clear()
    for method in config['methods']:
        series={}
        for depth in config['depths']:
            data=training.get(f'{method}_p{depth}',{}); starts=data.get('starts',[])
            if starts:
                trace=starts[data['best_start']]['trace']; series[f'p={depth}']=trace
        if series:
            fig=Figure(figsize=(8,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
            for label,trace in series.items():
                ax.plot([v['evaluation'] for v in trace],np.minimum.accumulate([v['native_cost'] for v in trace]),label=label)
            ax.set(title=f'{method}: best-start training',xlabel='Objective evaluations',ylabel='Best native expected cost')
            ax.legend(); ax.grid(alpha=.2); fig.savefig(out/f'03_training_{method}.png',dpi=150); fig.clear()
    evaluation=frame[frame.stage.isin(['ideal','noisy','real'])].copy()
    metrics=[('full_optimum_probability','Full optimum probability'),('mean_full_cost','Mean full pairwise cost'),('mean_mec','Mean MEC')]
    rows=[]
    for backend in config['backends']:
        subset=evaluation[((evaluation.backend==backend)&(evaluation.stage.isin(['noisy','real']))) | (evaluation.stage=='ideal')]
        for method in config['methods']:
            group=subset[subset.method==method]
            for metric,title in metrics[:2]:
                series={}
                for mode in ('ideal','noisy','real'):
                    one=group[group.stage==mode]
                    if one.empty: continue
                    means=[]; errors=[]
                    for depth in config['depths']:
                        vals=one.loc[one.depth==depth,metric].to_numpy(float)
                        means.append(float(np.mean(vals)) if len(vals) else np.nan)
                        errors.append(float(np.std(vals,ddof=1)) if len(vals)>1 else np.nan)
                    series[mode]={'mean':means,'error':errors}
                if series:
                    figure_file(path,f'04_depth_{backend}_{method}_{metric}.png',config['depths'],series,
                                f'{method} on {backend}: {title} by depth', 'QAOA depth p',title+' · repeat mean ± SD',
                                vline=float(np.isclose(full,full.min(),rtol=0,atol=1e-9).mean()) if metric=='full_optimum_probability' else None)
        focus=subset[subset.depth==champ_depth]
        for metric,title in metrics:
            fig=Figure(figsize=(7,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
            modes=[x for x in ('ideal','noisy','real') if x in set(focus.stage)]; width=.8/max(len(modes),1)
            for k,mode in enumerate(modes):
                vals=[]; lower=[]; upper=[]
                for method in config['methods']:
                    dat=focus[(focus.method==method)&(focus.stage==mode)]
                    if dat.empty: vals.append(np.nan); lower.append(0); upper.append(0); continue
                    val=float(dat[metric].mean()); vals.append(val)
                    if metric=='full_optimum_probability':
                        n=int(dat.shots.sum()); lo,hi=wilson(val*n,n); lower.append(val-lo); upper.append(hi-val)
                    else: lower.append(0); upper.append(0)
                ax.bar(np.arange(len(vals))+(k-(len(modes)-1)/2)*width,vals,width,label=mode,
                       yerr=np.array([lower,upper]) if metric=='full_optimum_probability' else None,capsize=2)
            ax.set_xticks(range(len(config['methods'])),config['methods'],rotation=15)
            ax.set(title=f'{backend}: {title} at p={champ_depth}',ylabel=title)
            if metric=='full_optimum_probability': ax.set_ylim(0,1.08)
            ax.grid(axis='y',alpha=.2); ax.legend(); fig.savefig(out/f'05_methods_{backend}_{metric}.png',dpi=150); fig.clear()
        for metric,title in [('depth','Total depth'),('two_qubit_depth','Two-qubit depth'),('two_qubit_gates','Two-qubit gates')]:
            series={m:[stats[f'{backend}|{m}|{p}'][metric] for p in config['depths']] for m in config['methods']}
            figure_file(path,f'06_resource_{backend}_{metric}.png',config['depths'],series,
                        f'{backend}: {title}', 'QAOA depth p',title)
        for stage in ('noisy','real'):
            local=tech[(tech.backend==backend)&(tech.stage==stage)] if not tech.empty else pd.DataFrame()
            if local.empty: continue
            base=local[local.technique=='raw_reference'].set_index('repeat')
            fig=Figure(figsize=(7,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
            labels=[name for name in ('readout','dd','twirling','zne') if name in set(local.technique)]
            for i,name in enumerate(labels):
                for _,row in local[local.technique==name].iterrows():
                    if row['repeat'] in base.index and pd.notna(row.mean_full_cost) and pd.notna(base.loc[row['repeat'],'mean_full_cost']):
                        ax.scatter(i,row.mean_full_cost-base.loc[row['repeat'],'mean_full_cost'],color='#237f94')
            ax.axhline(0,color='gray',ls='--'); ax.set_xticks(range(len(labels)),labels,rotation=15)
            ax.set(title=f'{backend} {stage}: technique cost change',ylabel='Mean cost − equal-budget raw'); ax.grid(axis='y',alpha=.2)
            fig.savefig(out/f'07_techniques_{backend}_{stage}.png',dpi=150); fig.clear()
            fig=Figure(figsize=(7,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
            ideal_cost=float(ideal[champ['method'],champ_depth]@full)
            for i,name in enumerate(labels):
                for _,row in local[local.technique==name].iterrows():
                    if row['repeat'] in base.index and pd.notna(row.mean_full_cost) and pd.notna(base.loc[row['repeat'],'mean_full_cost']):
                        difference=abs(row.mean_full_cost-ideal_cost)-abs(base.loc[row['repeat'],'mean_full_cost']-ideal_cost)
                        ax.scatter(i,difference,color='#237f94')
            ax.axhline(0,color='gray',ls='--'); ax.set_xticks(range(len(labels)),labels,rotation=15)
            ax.set(title=f'{backend} {stage}: change in cost error',ylabel='Absolute error change vs exact ideal'); ax.grid(axis='y',alpha=.2)
            fig.savefig(out/f'07_technique_error_{backend}_{stage}.png',dpi=150); fig.clear()
            zne=local[local.technique=='zne']
            if not zne.empty and 'factor_1_cost' in zne and 'factor_3_cost' in zne:
                fig=Figure(figsize=(6,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
                for _,row in zne.iterrows():
                    ax.plot([0,1,3],[row.mean_full_cost,row.factor_1_cost,row.factor_3_cost],marker='o',alpha=.7)
                exact=float(ideal[champ['method'],champ_depth]@full)
                ax.axhline(exact,ls='--',color='black',label='Exact ideal mean')
                ax.set(title=f'{backend} {stage}: ZNE',xlabel='Noise factor',ylabel='Estimated full cost',xticks=[0,1,3]); ax.legend(); ax.grid(alpha=.2)
                fig.savefig(out/f'08_zne_{backend}_{stage}.png',dpi=150); fig.clear()
    # Every raw outcome and every cost is saved; only displayed states are bounded.
    outcome_rows=[]
    for record in records:
        if record['stage'] not in ('ideal','noisy','real') or record['technique']!='raw': continue
        probs=normalize_counts(record['counts'],states)
        for state,prob,cost in zip(states,probs,full):
            outcome_rows.append({'stage':record['stage'],'backend':record['backend'],'method':record['method'],
                'depth':record['depth'],'repeat':record['repeat'],'state_snp_first':''.join(map(str,state)),
                'probability':float(prob),'full_cost':float(cost),'shots':record['shots']})
    freq=pd.DataFrame(outcome_rows)
    if not freq.empty:
        freq.to_csv(out/'outcome_frequencies.csv',index=False)
        depths=config['depths'] if config['all_distribution_depths'] else [champ_depth]
        for backend in config['backends']:
            for method in config['methods']:
                for depth in depths:
                    subset=freq[(freq.method==method)&(freq.depth==depth)&
                                ((freq.backend==backend)|(freq.stage=='ideal'))]
                    if subset.empty: continue
                    pivot=subset.groupby(['stage','state_snp_first'],as_index=False).probability.mean().pivot(index='state_snp_first',columns='stage',values='probability').fillna(0)
                    if len(pivot)>32:
                        keep=pivot.max(axis=1).nlargest(31).index; other=pivot.drop(keep).sum(); pivot=pivot.loc[keep]; pivot.loc['Other']=other
                    fig=Figure(figsize=(max(9,len(pivot)*.36),4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
                    modes=list(pivot.columns); width=.8/len(modes); x=np.arange(len(pivot))
                    for i,mode in enumerate(modes): ax.bar(x+(i-(len(modes)-1)/2)*width,pivot[mode],width,label=mode)
                    ax.set_xticks(x,pivot.index,rotation=90,fontsize=7); ax.set(title=f'{backend}: {method} p={depth} outputs',ylabel='Fraction of shots',xlabel='SNP 1 first')
                    ax.legend(); ax.grid(axis='y',alpha=.2); fig.savefig(out/f'09_outputs_{backend}_{method}_p{depth}.png',dpi=150); fig.clear()
                    grouped=subset.groupby(['stage','full_cost'],as_index=False).probability.sum()
                    grouped['probability']/=config['repeats']
                    costs=sorted(grouped.full_cost.unique()); fig=Figure(figsize=(8,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
                    for mode in modes:
                        vals=grouped[grouped.stage==mode].set_index('full_cost').reindex(costs).probability.fillna(0)
                        ax.plot(costs,vals,marker='o',label=mode)
                    ax.set(title=f'{backend}: {method} p={depth} cost distribution',xlabel='Full pairwise cost',ylabel='Fraction of shots')
                    ax.legend(); ax.grid(alpha=.2); fig.savefig(out/f'10_costs_{backend}_{method}_p{depth}.png',dpi=150); fig.clear()
    # Batch elapsed times include all circuits in the batch, not individual circuit latency.
    times=pd.DataFrame([{'stage':r['stage'],'backend':r['backend'],'repeat':r['repeat'],
                         'batch_wall_seconds':r.get('batch_wall_seconds'),'batch_circuits':r.get('batch_circuits'),
                         'job_id':r.get('job_id')} for r in records])
    if not times.empty:
        times=times.drop_duplicates(); times.to_csv(out/'batch_times.csv',index=False)
        for backend in config['backends']:
            data=times[(times.backend==backend)&times.batch_wall_seconds.notna()]
            if not data.empty:
                fig=Figure(figsize=(7,4),layout='constrained'); FigureCanvasAgg(fig); ax=fig.subplots()
                for i,stage in enumerate(sorted(data.stage.unique())):
                    vals=data[data.stage==stage].batch_wall_seconds; ax.scatter(np.full(len(vals),i),vals,label=stage)
                ax.set_xticks(range(len(sorted(data.stage.unique()))),sorted(data.stage.unique())); ax.set(title=f'{backend}: batch wall times',ylabel='Seconds; hardware includes queue')
                ax.grid(axis='y',alpha=.2); fig.savefig(out/f'11_execution_time_{backend}.png',dpi=150); fig.clear()
    # Tabular metric data are retained beside every figure.
    frame.to_csv(out/'metric_plot_data.csv',index=False)
    if not tech.empty: tech.to_csv(out/'technique_plot_data.csv',index=False)
    print('Plots saved:',len(list(out.glob('*.png'))),flush=True)
    for method in config['methods']:
        series={}
        for depth in config['depths']:
            info=training.get(f'{method}_p{depth}',{})
            if info.get('wall_seconds') is not None: series.setdefault(method,[]).append(info['wall_seconds'])
        if series and len(series[method])==len(config['depths']):
            figure_file(path,f'12_training_time_{method}.png',config['depths'],series,
                        f'{method}: training time','QAOA depth p','Wall seconds, all starts')
    for backend in config['backends']:
        series={}
        for method in config['methods']:
            vals=[stats[f'{backend}|{method}|{p}'].get('compile_wall_seconds') for p in config['depths']]
            if all(v is not None for v in vals): series[method]=vals
        if series:
            figure_file(path,f'13_compile_time_{backend}.png',config['depths'],series,
                        f'{backend}: compilation time','QAOA depth p','Wall seconds')
        for method in config['methods']:
            subset=evaluation[(evaluation.method==method)&(evaluation.backend==backend)&(evaluation.stage.isin(['noisy','real']))]
            for metric,title in [('total_variation_from_ideal','Distance from ideal'),('mean_full_cost','Mean full cost')]:
                series={}
                for stage in ('noisy','real'):
                    part=subset[subset.stage==stage]
                    if part.empty: continue
                    vals=[]
                    for depth in config['depths']:
                        x=part.loc[part.depth==depth,metric]
                        value=float(x.mean()) if len(x) else np.nan
                        if metric=='mean_full_cost' and len(x): value -= float(ideal[method,depth]@full)
                        vals.append(value)
                    series[stage]=vals
                if series:
                    figure_file(path,f'14_noise_{backend}_{method}_{metric}.png',config['depths'],series,
                                f'{backend} {method}: {title}', 'QAOA depth p',
                                'Total variation' if metric.startswith('total_') else 'Mean cost − exact ideal')


## 7. Offline support for earlier runs
Older result folders are read locally. Their original data and manifests remain unchanged; regenerated plots are written under their `plots` folder.

In [ ]:
"""Recreate comparison figures from a saved backend study, without running jobs."""

import json
from pathlib import Path

from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.ticker import PercentFormatter
from matplotlib.colors import PowerNorm
import numpy as np
import pandas as pd

MODE_COLORS = {'ideal': '#337CB5', 'noisy': '#D9902F', 'real': '#B84F61'}
MODE_LABELS = {'ideal': 'Ideal sample', 'noisy': 'Calibrated noise', 'real': 'Real device'}


def _subplots(*args, figsize=None, constrained_layout=False, **kwargs):
    fig = Figure(figsize=figsize, layout='constrained' if constrained_layout else None)
    FigureCanvasAgg(fig)
    return fig, fig.subplots(*args, **kwargs)


def _name(name):
    return name.replace('standard', 'Standard').replace('multi', 'Multi-angle').replace('_full_', ' · full · ').replace('_forest_', ' · forest · ')


def _groups(frame, backends, methods):
    return [(backend, method) for backend in backends for method in methods
            if ((frame.backend == backend) & (frame.method == method)).any()]


def _bars(ax, frame, metric, groups, modes, *, probability=False, errors=False, baseline=None):
    width = 0.75 / max(len(modes), 1)
    for k, mode in enumerate(modes):
        y, vals, err_low, err_high = [], [], [], []
        for i, (backend, method) in enumerate(groups):
            match = frame[(frame.backend == backend) & (frame.method == method) & (frame['mode'] == mode)]
            if match.empty or metric not in match or pd.isna(match.iloc[0][metric]):
                continue
            row = match.iloc[0]; y.append(i + (k-(len(modes)-1)/2)*width)
            vals.append(float(row[metric]))
            if errors:
                # Binomial shot uncertainty only; this does not include model/calibration uncertainty.
                p, shots = float(row[metric]), int(row.shots)
                z = 1.96
                denominator = 1 + z*z/shots
                center = (p + z*z/(2*shots))/denominator
                half = z*np.sqrt(p*(1-p)/shots + z*z/(4*shots*shots))/denominator
                err_low.append(max(0., p-(center-half)))
                err_high.append(max(0., (center+half)-p))
        if vals:
            ax.barh(y, vals, height=width*.9, color=MODE_COLORS[mode], label=MODE_LABELS[mode],
                    xerr=np.array([err_low,err_high]) if errors else None, error_kw={'capsize': 2, 'linewidth': .8})
    if baseline is not None:
        ax.axvline(baseline, color='#263238', linestyle='--', linewidth=1, label='Exact minimum')
    ax.set_yticks(range(len(groups)), [f'{b.replace("ibm_", "").title()}  |  {_name(m)}' for b,m in groups])
    ax.invert_yaxis(); ax.grid(axis='x', alpha=.18); ax.set_axisbelow(True)
    if probability:
        ax.set_xlim(0,1.05); ax.xaxis.set_major_formatter(PercentFormatter(1))


def _save(fig, folder, name, paths):
    fig.savefig(folder/name, dpi=150, bbox_inches='tight', facecolor='white')
    fig.clear(); paths.append(folder/name)


def _read(path):
    return json.loads(path.read_text(encoding='utf-8'))


def plot_legacy_run(run_dir):
    """Create figures and frequency tables using already saved data; no backend access."""
    # Import lazily to reuse the study's validated scoring and bit-order conventions.

    run = Path(run_dir)
    manifest = _read(run/'manifest.json')
    reference = _read(run/'references.json')
    counts = _read(run/'counts.json')
    training = _read(run/'training.json')
    results = pd.read_csv(run/'results.csv')
    screening = pd.read_csv(run/'screening.csv')
    _, _, _, states, full, mec = problem(manifest['reads'])
    state_labels = [''.join(map(str, s)) for s in states]
    state_index = {state: i for i, state in enumerate(state_labels)}
    backends = list(dict.fromkeys(results.backend))
    methods = manifest.get('selected_methods') or list(dict.fromkeys(results.method))
    methods = [m for m in methods if m in set(results.method)]
    modes = [m for m in ('ideal', 'noisy', 'real') if m in set(results['mode'])]
    groups = _groups(results, backends, methods)
    out = run/'plots'; out.mkdir(exist_ok=True)
    paths = []

    # The screening plot covers every candidate; final evaluation may contain only a subset.
    candidate_order = list(dict.fromkeys(screening.method))
    screen_groups = _groups(screening, backends, candidate_order)
    screen = screening.copy(); screen['mode'] = 'noisy'
    fig, axes = _subplots(1, 2, figsize=(14, max(5, .36*len(screen_groups)+2)), constrained_layout=True)
    for ax, col, title, prob in zip(axes, ['full_optimum_probability','mean_full_cost'],
                                     ['Full-optimum frequency', 'Mean full pairwise cost'], [True, False]):
        _bars(ax, screen, col, screen_groups, ['noisy'], probability=prob, errors=prob,
              baseline=None if prob else reference['full_min'])
        ax.set_title(title); ax.set_xlabel('Proportion of shots' if prob else 'Cost')
    axes[1].set_yticklabels([])
    fig.suptitle('Screening: all candidates under each device noise model', fontsize=14, fontweight='bold')
    _save(fig, out, '01_screening.png', paths)

    fig, axes = _subplots(2, 2, figsize=(15, max(7, .42*len(groups)+4)), constrained_layout=True)
    panels = [('full_optimum_probability','Full pairwise optimum frequency',True,None),
              ('mec_optimum_probability','MEC optimum frequency',True,None),
              ('mean_full_cost','Mean full pairwise cost',False,reference['full_min']),
              ('mean_mec','Mean MEC',False,reference['mec_min'])]
    for ax, (col,title,prob,baseline) in zip(axes.flat, panels):
        _bars(ax, results, col, groups, modes, probability=prob, errors=prob, baseline=baseline)
        ax.set_title(title); ax.set_xlabel('Proportion of shots' if prob else 'Cost')
    for ax in axes[:,1]: ax.set_yticklabels([])
    handles, labels = axes[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='outside lower center',ncol=len(modes),frameon=False)
    fig.suptitle('Final evaluation · whiskers show 95% Wilson shot intervals', fontsize=14, fontweight='bold')
    _save(fig, out, '02_quality.png', paths)

    # Each resource describes the compiled circuit, independent of execution mode.
    resources = results.drop_duplicates(['backend','method']).copy(); resources['mode'] = 'noisy'
    fig, axes = _subplots(1, 3, figsize=(15,max(4,.35*len(groups)+2)), constrained_layout=True)
    for ax, (col,title) in zip(axes, [('depth','Total depth'),('two_qubit_depth','Two-qubit depth'),('two_qubit_gates','Two-qubit gates')]):
        _bars(ax, resources, col, groups, ['noisy']); ax.set_title(title); ax.set_xlabel('Compiled circuit')
    for ax in axes[1:]: ax.set_yticklabels([])
    fig.suptitle('Hardware-compiled circuit resources · one circuit per device and method', fontsize=14, fontweight='bold')
    _save(fig, out, '03_resources.png', paths)

    fig, axes = _subplots(1, 2, figsize=(14,max(4,.38*len(groups)+2)), constrained_layout=True)
    _bars(axes[0], results, 'total_variation_from_ideal', groups, modes, probability=True)
    axes[0].set_title('Distance from exact ideal distribution'); axes[0].set_xlabel('Total variation distance')
    raw_corrected = results[results['mode'].isin(['noisy','real'])].copy()
    # Use color for raw versus corrected; label includes device and mode.
    quality_groups = [(r.backend,r.method,r['mode']) for _,r in raw_corrected.iterrows()]
    for i, (backend,method,mode) in enumerate(quality_groups):
        row = raw_corrected[(raw_corrected.backend==backend)&(raw_corrected.method==method)&(raw_corrected['mode']==mode)].iloc[0]
        axes[1].barh(i-.18, row.mean_full_cost, height=.34, color='#8AA6BA', label='Raw' if i == 0 else None)
        if pd.notna(row.readout_corrected_mean_full_cost):
            axes[1].barh(i+.18, row.readout_corrected_mean_full_cost, height=.34, color='#D9902F', label='Readout corrected' if i == 0 else None)
    axes[1].set_yticks(range(len(quality_groups)), [f'{b.replace("ibm_", "").title()} | {_name(m)} | {mode}' for b,m,mode in quality_groups], fontsize=8)
    axes[1].invert_yaxis(); axes[1].axvline(reference['full_min'],color='#263238',linestyle='--',label='Exact minimum')
    axes[1].set_title('Readout correction estimates mean cost'); axes[1].set_xlabel('Full pairwise cost'); axes[1].grid(axis='x',alpha=.18)
    axes[1].legend(fontsize=8)
    fig.suptitle('Distribution error and readout correction', fontsize=14, fontweight='bold')
    _save(fig, out, '04_noise_and_readout.png', paths)

    # Seconds are reported separately: each mode has different execution and calibration overhead.
    fig, axes = _subplots(1, 2, figsize=(14,max(4,.38*len(groups)+2)), constrained_layout=True)
    methods_with_time = [m for m in methods if training.get(m,{}).get('wall_seconds') is not None]
    axes[0].barh([_name(m) for m in methods_with_time], [training[m]['wall_seconds'] for m in methods_with_time], color='#527B8A')
    axes[0].invert_yaxis(); axes[0].set_title('Local training'); axes[0].set_xlabel('Wall seconds · all starts')
    if not methods_with_time:
        axes[0].text(.5,.5,'Not recorded in this run',ha='center',transform=axes[0].transAxes)
    _bars(axes[1], resources, 'compile_wall_seconds', groups, ['noisy'])
    if 'compile_wall_seconds' not in resources or resources['compile_wall_seconds'].isna().all():
        axes[1].text(.5,.5,'Not recorded in this run',ha='center',transform=axes[1].transAxes)
    axes[1].set_title('Compilation per device'); axes[1].set_xlabel('Wall seconds')
    fig.suptitle('Classical preparation time', fontsize=14, fontweight='bold')
    _save(fig, out, '05_preparation_time.png', paths)
    if 'execution_wall_seconds' in results and results.execution_wall_seconds.notna().any():
        fig, axes = _subplots(1,len(modes),figsize=(6*len(modes),max(4,.38*len(groups)+2)),squeeze=False,constrained_layout=True)
        for ax,mode in zip(axes[0],modes):
            _bars(ax, results, 'execution_wall_seconds', groups, [mode])
            ax.set_title(MODE_LABELS[mode]); ax.set_xlabel('Wall seconds')
        fig.suptitle('Execution wall time · noise/real include two calibrations; real includes queue', fontsize=13, fontweight='bold')
        _save(fig, out, '06_execution_time.png', paths)

    # Full frequency tables use SNP-first labels. Qiskit count strings are reversed.
    frequencies = []; cost_frequencies = []
    exact = dict(reference.get('ideal_probabilities', {}))
    for method in methods:
        # Older runs stored the frozen circuit but not the exact probabilities.
        circuit_file = run/f'{method}_logical.qpy'
        if method not in exact and circuit_file.exists():
            from qiskit import qpy
            from qiskit.quantum_info import Statevector
            with circuit_file.open('rb') as stream:
                circuit = qpy.load(stream)[0]
            vector = Statevector.from_instruction(circuit.remove_final_measurements(inplace=False)).probabilities()
            exact[method] = {label: float(vector[sum(int(bit) << i for i, bit in enumerate(state))])
                             for label,state in zip(state_labels,states)}
        distributions = []
        if method in exact:
            distributions.append(('Exact ideal', np.array([float(exact[method].get(s,0)) for s in state_labels])))
        for backend in backends:
            for mode in modes:
                entry = counts.get(f'{backend}:{method}:{mode}')
                if not entry: continue
                samples = entry['counts']; total = sum(samples.values())
                if total <= 0: raise ValueError('A saved result has no shots.')
                probs = np.zeros(len(states))
                for key,count in samples.items():
                    label = key.replace(' ','')[::-1]
                    if label not in state_index: raise ValueError(f'Unexpected count bitstring: {key}')
                    probs[state_index[label]] += count/total
                distributions.append((f'{backend.replace("ibm_", "").title()} · {MODE_LABELS[mode]}',probs))
                for label,prob in zip(state_labels,probs):
                    frequencies.append({'backend':backend,'method':method,'mode':mode,'state_snp_first':label,'probability':prob,'shots':total})
        if method in exact:
            for label,prob in zip(state_labels,distributions[0][1]):
                frequencies.append({'backend':'','method':method,'mode':'ideal_exact','state_snp_first':label,'probability':prob,'shots':None})
        if not distributions: continue
        matrix=np.stack([x[1] for x in distributions])
        if len(state_labels) <= 32:
            chosen=np.arange(len(state_labels)); display_labels=state_labels; shown=matrix
        else:
            chosen=np.argsort(-matrix.max(axis=0),kind='stable')[:31]
            display_labels=[state_labels[i] for i in chosen]+['Other']
            shown=np.column_stack([matrix[:,chosen], np.maximum(0, 1-matrix[:,chosen].sum(axis=1))])
        fig, ax = _subplots(figsize=(max(10,.40*len(display_labels)+3),max(3,.55*len(distributions)+1.8)),constrained_layout=True)
        im=ax.imshow(shown,aspect='auto',cmap='Blues',norm=PowerNorm(gamma=.45,vmin=0,vmax=max(float(shown.max()),.01)))
        ax.set_xticks(range(len(display_labels)), [s+'*' if s in reference['full_minimizers'] else s for s in display_labels], rotation=90,fontsize=8)
        ax.set_yticks(range(len(distributions)), [name for name,_ in distributions])
        ax.set_xlabel('Output state · SNP 1 first · * full optimum')
        ax.set_title(f'Final output frequencies · {_name(method)}',fontweight='bold')
        color=fig.colorbar(im,ax=ax,shrink=.8); color.ax.yaxis.set_major_formatter(PercentFormatter(1)); color.set_label('Fraction of shots')
        _save(fig,out,f'07_outcomes_{method}.png',paths)

        # Aggregate every output, including tail states, by its full pairwise cost.
        cost_values=np.unique(full)
        if len(cost_values)>30:
            edges=np.linspace(float(full.min()),float(full.max())+1e-9,21)
            bin_index=np.clip(np.digitize(full,edges)-1,0,len(edges)-2)
            x=(edges[:-1]+edges[1:])/2; lower=edges[:-1]; upper=edges[1:]; xlabel='Full pairwise cost (20 bins)'
        else:
            bin_index=np.searchsorted(cost_values,full); x=cost_values; lower=upper=cost_values; xlabel='Full pairwise cost'
        fig, ax=_subplots(figsize=(max(9,.65*len(x)+3),4.5),constrained_layout=True)
        width=.8/len(distributions)
        for index,(label,probs) in enumerate(distributions):
            values=np.bincount(bin_index,weights=probs,minlength=len(x))
            positions=np.arange(len(x))+(index-(len(distributions)-1)/2)*width
            ax.bar(positions,values,width=width,label=label)
            for cost,low,high,prob in zip(x,lower,upper,values):
                cost_frequencies.append({'method':method,'series':label,'cost_or_bin_midpoint':float(cost),
                                         'bin_lower':float(low),'bin_upper':float(high),'probability':float(prob)})
        ax.set_xticks(range(len(x)),[f'{v:g}' for v in x],rotation=45 if len(x)>12 else 0)
        ax.set(title=f'Whole cost distribution · {_name(method)}',xlabel=xlabel,ylabel='Fraction of shots')
        ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.grid(axis='y',alpha=.2); ax.legend(fontsize=8,ncol=2)
        _save(fig,out,f'08_cost_distribution_{method}.png',paths)
    pd.DataFrame(frequencies).to_csv(out/'outcome_frequencies.csv',index=False)
    pd.DataFrame(cost_frequencies).to_csv(out/'cost_frequencies.csv',index=False)
    return paths


In [ ]:
def load_run(path):
    path=Path(path); manifest=json.loads((path/'manifest.json').read_text(encoding='utf8'))
    if manifest.get('schema_version') != 2:
        images=plot_legacy_run(path)
        print('Loaded legacy run; absent historical values are not recorded.')
        return path,pd.read_csv(path/'ranking.csv'),pd.read_csv(path/'results.csv'),pd.DataFrame()
    config=manifest['config']; reads,a,d,states,full,mec=problem(config['reads'])
    training=json.loads((path/'training.json').read_text(encoding='utf8'))
    stats=json.loads((path/'compiled_stats.json').read_text(encoding='utf8'))
    ideal_json=json.loads((path/'ideal_probabilities.json').read_text(encoding='utf8'))
    ideal={(method,int(depth)):np.array(values) for key,values in ideal_json.items() for method,depth in [key.split('|')]}
    frame=pd.read_csv(path/'results.csv'); tech_file=path/'techniques.csv'
    tech=pd.read_csv(tech_file) if tech_file.exists() else pd.DataFrame()
    plot_study(path,manifest,states,full,mec,training,stats,ideal,frame,tech,read_records(path))
    return path,pd.read_csv(path/'ranking.csv'),frame,tech


## 8. Screening, independent evaluation and optional hardware
The following function coordinates local training, device-calibrated screening, champion selection, independent evaluation and optional real jobs. It checkpoints every circuit result and protects completed data from plotting errors.

In [ ]:
def run_study(root, config, backend_objects=None, service_override=None):
    validate_config(config)
    path, manifest = initialize_run(root, config)
    if config['mode'] == 'load':
        return load_run(path)
    if config['mode'] == 'resume':
        previous = manifest['config']
        ignored = {'mode','run_folder','run_real','output_dir'}
        changed = [key for key in previous if key not in ignored and previous.get(key) != config.get(key)]
        if changed:
            raise ValueError(f'Resume settings differ from the saved run: {changed}')
    if config['mode']=='resume' and config['run_real']:
        manifest['config']['run_real']=True
        write_manifest(path,manifest)
    reads,a,d,states,full,mec = problem(config['reads'])
    forest = forest_j(a,d)
    j = forest if config['prune_rule']=='mstforest' else (d-a)/2
    if config['prune_rule'] not in ('mstforest','none'):
        raise ValueError('PRUNE_RULE must be mstforest or none.')
    native_offset = float(np.where(j != 0,(a+d)/2,0).sum()) if config['prune_rule']=='mstforest' else float((a+d).sum()/2)
    native = native_scores(states,j,native_offset)
    best, local_bits, used, hit = greedy_local_search(j,native_offset,config['local_budget'],config['local_restarts'],np.random.default_rng(config['local_seed']))
    warm_seed = local_bits[0].astype(float)
    save_json(path/'problem.json',{'reads':reads.tolist(),'same':a.tolist(),'opposite':d.tolist(),
        'native_j':j.tolist(),'native_offset':native_offset,'warm_seed':warm_seed.tolist(),
        'local_best':best.tolist(),'local_used':used.tolist(),
        'active_components':[sorted(c) for c in nx.connected_components(nx.from_numpy_array(j+j.T))],
        'native_minimizers_preserve_full':bool(np.all(np.isclose(full[np.isclose(native,native.min(),rtol=0,atol=1e-9)],full.min(),rtol=0,atol=1e-9))),
        'full_min':float(full.min()),
        'mec_min':float(mec.min()),'full_minimizers':[''.join(map(str,s)) for s in states[np.isclose(full,full.min(),atol=1e-9,rtol=0)]]})
    try:
        if config['mode']=='resume' and any(jb['status']!='completed' for jb in manifest['jobs']):
            service = service_override or ibm_service(root)
            recover_submitted(path,manifest,service)
        service,backends,simulators = setup_backends(root,path,manifest,config,backend_objects)
        service = service_override or service
        circuits={}; training={}; ideal={}; stats={}
        for method in config['methods']:
            for depth in config['depths']:
                ident=f'{method}_p{depth}'; logical_file=path/'circuits'/f'{ident}_logical.qpy'
                train_file=path/'circuits'/f'{ident}_training.json'
                if logical_file.exists() and train_file.exists():
                    circuit=load_circuit(logical_file); info=json.loads(train_file.read_text(encoding='utf8'))
                else:
                    print('Training',ident,flush=True)
                    template,params=make_template(method,j,depth,warm_seed,config['epsilon'])
                    iterations=config['multi_iterations'] if method=='multi-angle' else config['standard_iterations']
                    circuit,info=fit_template(template,params,states,native,full,iterations,config['starts'],config['training_seed'])
                    circuit.measure_all(); save_circuit(logical_file,circuit); save_json(train_file,info)
                circuits[method,depth]=circuit; training[ident]=info
                ideal[method,depth]=ideal_probabilities(circuit,states)
        save_json(path/'training.json',training)
        save_json(path/'ideal_probabilities.json',{f'{m}|{p}':v.tolist() for (m,p),v in ideal.items()})
        compiled={}
        for name,backend in backends.items():
            pm=generate_preset_pass_manager(backend=backend,optimization_level=config['optimization_level'],seed_transpiler=config['transpile_seed'])
            for (method,depth),circuit in circuits.items():
                fname=path/'circuits'/f'{name}_{method}_p{depth}_compiled.qpy'
                if fname.exists():
                    qc=load_circuit(fname); elapsed=None
                else:
                    began=time.perf_counter(); qc=pm.run(circuit); elapsed=time.perf_counter()-began
                    save_circuit(fname,qc)
                compiled[name,method,depth]=qc
                stats[f'{name}|{method}|{depth}']={**circuit_stats(qc),'compile_wall_seconds':elapsed}
        old_stats=path/'compiled_stats.json'
        if old_stats.exists():
            previous=json.loads(old_stats.read_text(encoding='utf8'))
            for key,val in stats.items():
                if val['compile_wall_seconds'] is None and key in previous:
                    val['compile_wall_seconds']=previous[key].get('compile_wall_seconds')
        save_json(old_stats,stats)
        for name,simulator in simulators.items():
            print('Screening',name,flush=True)
            for repeat in range(config['screen_repeats']):
                tasks=[task_for(path,compiled[name,m,p],'screen',name,m,p,repeat,'raw','target',config['screen_shots'])
                       for m,p in circuits]
                run_local_tasks(path,simulator,tasks,config)
        screen,_,_=summarize_records(path,states,full,mec,ideal,a,d)
        screen=screen[screen.stage=='screen'].copy()
        screen.to_csv(path/'screening.csv',index=False)
        ranking=(screen.groupby(['method','depth'],as_index=False)
                 .agg(full_optimum_probability=('full_optimum_probability','mean'),mean_full_cost=('mean_full_cost','mean')))
        ranking['mean_two_qubit_gates']=[np.mean([stats[f'{name}|{row.method}|{row.depth}']['two_qubit_gates']
                                                for name in backends]) for row in ranking.itertuples()]
        ranking=ranking.sort_values(['full_optimum_probability','mean_full_cost','mean_two_qubit_gates'],
                                    ascending=[False,True,True]).reset_index(drop=True)
        ranking.to_csv(path/'ranking.csv',index=False)
        champion={'method':str(ranking.iloc[0]['method']),'depth':int(ranking.iloc[0]['depth'])}
        if manifest.get('champion') and manifest['champion'] != champion:
            raise RuntimeError('Saved champion differs from recovered screening; refusing to mix runs.')
        manifest['champion']=champion; manifest['selection_rule']='mean noisy full optimum; tie-break full cost, then 2q gates'
        manifest['status']='screened'; write_manifest(path,manifest)
        print('Selected champion:',champion,'(exploratory screening)',flush=True)
        ideal_sim=AerSimulator(max_parallel_threads=1)
        for repeat in range(config['repeats']):
            tasks=[task_for(path,c,'ideal','ideal',m,p,repeat,'raw','target',config['shots']) for (m,p),c in circuits.items()]
            run_local_tasks(path,ideal_sim,tasks,config)
        hardware={}
        for name,backend in backends.items():
            print('Evaluating',name,flush=True)
            for repeat in range(config['repeats']):
                raw=[task_for(path,compiled[name,m,p],'noisy',name,m,p,repeat,'raw','target',config['shots'])
                     for m,p in circuits]
                run_local_tasks(path,simulators[name],raw,config)
                m,p=champion['method'],champion['depth']
                arms,skips=prepare_techniques(compiled[name,m,p],backend,simulators[name],config,repeat)
                for technique,reason in skips.items():
                    skip={'backend':name,'repeat':repeat,'technique':technique,'reason':reason}
                    if skip not in manifest['skips']: manifest['skips'].append(skip)
                write_manifest(path,manifest)
                noisy=[]; remote=[]
                for technique,variants in arms.items():
                    for variant,qc,shots in variants:
                        noisy.append(task_for(path,qc,'noisy',name,m,p,repeat,technique,variant,shots))
                        if config['run_real']:
                            remote.append(task_for(path,qc,'real',name,m,p,repeat,technique,variant,shots))
                run_local_tasks(path,simulators[name],noisy,config)
                if config['run_real']:
                    raw_real=[task_for(path,compiled[name,x,y],'real',name,x,y,repeat,'raw','target',config['shots'])
                              for x,y in circuits]
                    hardware[name,repeat]=raw_real+remote
        if config['run_real']:
            if service is None: raise RuntimeError('A Runtime service is required for hardware execution.')
            preview=[{'backend':name,'repeat':repeat,'pubs':len(tasks),'shots':sum(t['shots'] for t in tasks)}
                     for (name,repeat),tasks in hardware.items()]
            save_json(path/'hardware_preview.json',preview)
            print('Hardware workload:',preview,'total shots:',sum(x['shots'] for x in preview),flush=True)
            for (name,repeat),tasks in hardware.items():
                run_remote_tasks(path,manifest,service,backends[name],tasks)
        frame,tech,records=summarize_records(path,states,full,mec,ideal,a,d)
        manifest.pop('error',None)
        manifest['status']='complete'; manifest['completed_utc']=datetime.now(timezone.utc).isoformat()
        write_manifest(path,manifest)
        try:
            plot_study(path,manifest,states,full,mec,training,stats,ideal,frame,tech,records)
            manifest.pop('plot_error',None)
        except Exception as exc:
            manifest['plot_error']=f'{type(exc).__name__}: {exc}'
            print('Results saved; plotting failed:',manifest['plot_error'],flush=True)
        write_manifest(path,manifest)
        return path,ranking,frame,tech
    except Exception as exc:
        manifest['status']='failed'; manifest['error']=f'{type(exc).__name__}: {exc}'; write_manifest(path,manifest)
        print('Partial data saved in',path,flush=True)
        raise


In [ ]:
validate_config(CONFIG)
print('Planned workload (before unsupported-arm skips):')
display(pd.DataFrame([workload(CONFIG)]).T.rename(columns={0:'count'}))
run_dir, ranking, results, technique_results = run_study(ROOT, CONFIG, backend_objects=BACKEND_OVERRIDES)
print('Saved run:', run_dir)
display(ranking)
display(results)
if not technique_results.empty: display(technique_results)

## 9. Results and plots
Success means matching any exact minimum of the full pairwise objective; it is not necessarily biological accuracy. Local noise approximates device behavior. Real-device differences include calibration and implementation as well as topology. Execution times can include calibration, network and queue overhead.

In [ ]:
if SHOW_PLOTS:
    for image_path in sorted((run_dir/'plots').glob('*.png')):
        display(Image(filename=str(image_path)))

## 10. Recover a submitted job
Use `MODE="resume"` with `RUN_FOLDER` and the original settings. Submitted job IDs and their PUB mapping are recovered before further work; a partial job is never submitted again. If calibration changed, retrieved results remain saved and a new run is required for comparable local simulation. Results under `bench/results` are Git-ignored; back them up separately.